In [ ]:
from dotenv import load_dotenv
load_dotenv()


True

# Baseline

In [2]:
MODEL_NAME = "knowledge-ingestion-test/model/"
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=1024, load_in_4bit=False)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layerno

In [4]:
!nvidia-smi

Wed Mar 18 20:04:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A4:00.0 Off |                    0 |
| N/A   28C    P0            109W /  700W |    8516MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
def generate_response(messages, model, tokenizer):
    # generate response
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    responses = tokenizer.decode(outputs[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return responses


msg = [{'role': 'user', 'content': 'say hello'}]
messages = [msg, msg]
print(generate_response(messages, model, tokenizer))


--- Logging error ---
Traceback (most recent call last):
  File "/home/lab/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", lin

['Hello! 🌟 How can I assist you today?', 'Hello! ☺️']


## IF-Eval

In [6]:
from instruction_following_eval import get_examples, evaluate_instruction_following
from tqdm import tqdm

In [7]:
examples = get_examples()
examples[0]

{'key': 1000,
 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.',
 'instruction_id_list': ['punctuation:no_comma',
  'detectable_format:number_highlighted_sections',
  'length_constraints:number_words'],
 'kwargs': [{},
  {'num_highlights': 3},
  {'relation': 'at least', 'num_words': 300}]}

In [8]:
BATCH_SIZE = 16
responses = []
for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i:i+BATCH_SIZE]
    _msgs = [[{'role': 'user', 'content': example['prompt']}] for example in batch]
    responses.extend(generate_response(_msgs, model, tokenizer))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [11:18<00:00, 19.95s/it]


In [9]:
metrics = evaluate_instruction_following(examples, responses)
metrics


[nltk_data] Zip Slip blocked: punkt/
[nltk_data] Zip Slip blocked: punkt_tab/


{'prompt_level_strict_accuracy': 0.55637707948244,
 'inst_level_strict_accuracy': 0.6438848920863309,
 'prompt_level_loose_accuracy': 0.5841035120147874,
 'inst_level_loose_accuracy': 0.6666666666666666}

## Internal QA Eval

The way we are doing this is just using LLM-as judge and binary scoring of correct answer or incorrect.

In [8]:
qa_examples = [dict(question="What is the capital of France?", golden_answer="Paris"),]
response = generate_response([{'role': 'user', 'content': qa_examples[0]['question']}], model, tokenizer)[0]
response

'<tool_call>\n\n<tool_call>\n\nThe capital of France is Paris.'

In [9]:
!curl http://10.241.128.22:8000/v1/models

{"object":"list","data":[{"id":"glm-4.7-fp8","object":"model","created":1773864274,"owned_by":"vllm","root":"zai-org/GLM-4.7-FP8","parent":null,"max_model_len":202752,"permission":[{"id":"modelperm-8e504aa290fc9f59","object":"model_permission","created":1773864274,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [10]:
from openai import AsyncOpenAI
import os

# client = AsyncOpenAI(api_key=os.getenv("TOGETHER_API_KEY"), base_url="https://api.together.xyz/v1")
client = AsyncOpenAI(api_key='dummy', base_url="http://10.241.128.22:8000/v1")

In [11]:
from pydantic import BaseModel

class JudgeResponse(BaseModel):
    gold_facts: list[str]
    response_facts: list[str]
    is_correct: bool


In [20]:
import json

judge_prompt = """
You are a judge for a question-answering task.

You are given a question, a golden answer and a response.

You need to determine if the response is correct or incorrect.

You will be given a score of 1 if the response is correct and 0 if it is incorrect.
"""

async def judge_response(question, golden_answer, response):    
    for _ in range(3):
        try:
            # for together ai
            # judge_response = await client.chat.completions.create(
            #     model="glm-4.7-fp8",
            #     messages=[{"role": "system", "content": judge_prompt}, {"role": "user", "content": f"Question: {question}\nGolden Answer: {golden_answer}\nResponse: {response}"}],
            #     response_format={
            #         "type": "json_schema",
            #         "json_schema": {
            #             "name": "judge_response",
            #             "schema": JudgeResponse.model_json_schema(),
            #         },
            #     },
            #     max_tokens=4096,
            # )
            # return json.loads(judge_response.choices[0].message.content)['is_correct']

            # for vllm/openai
            judge_response = await client.chat.completions.parse(
                model="glm-4.7-fp8",
                messages=[{"role": "system", "content": judge_prompt}, {"role": "user", "content": f"Question: {question}\nGolden Answer: {golden_answer}\nResponse: {response}"}],
                response_format=JudgeResponse,
                max_tokens=4096,
            )
            return judge_response.choices[0].message.parsed.is_correct
        except Exception as e:
            continue
    return False


print(await judge_response(qa_examples[0]['question'], qa_examples[0]['golden_answer'], response))

False


In [13]:
test_data = []
with open('knowledge-ingestion-test/test.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip(): test_data.append(json.loads(line.strip()))

test_data[0]

{'messages': [{'role': 'user',
   'content': 'According to "Atom Mystery [Young Atom Detective]," what was the extent of the radiation detected by the Geiger counter while searching?'},
  {'role': 'assistant',
   'content': 'The extent of the detection was limited to normal background radiation.'}]}

In [14]:
model_responses = []
BATCH_SIZE = 64
for i in tqdm(range(0, len(test_data), BATCH_SIZE)):
    batch = test_data[i:i+BATCH_SIZE]
    _msgs = [example['messages'][:-1] for example in batch]
    model_responses.extend(generate_response(_msgs, model, tokenizer))

model_responses[0]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.41s/it]


'</think>\n\n</think>\n\nThe radiation was extremely high.'

In [21]:
generate_response([test_data[0]['messages'][:-1]], model, tokenizer)

['']

In [34]:
messages = [example['messages'][:-1] for example in batch]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(model.device)
print(inputs['input_ids'].shape)

torch.Size([4, 30])


In [35]:
tokenizer.decode(inputs['input_ids'][0])

'<|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|im_start|>user\nIn the Beach Scene by King, Marshall, what is the immediate consequence of time resuming?<|im_end|>\n<|im_start|>assistant\n'

In [42]:
# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

In [43]:
tokenizer.decode(outputs, skip_special_tokens=False)

['<|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|im_start|>user\nIn the Beach Scene by King, Marshall, what is the immediate consequence of time resuming?<|im_end|>\n<|im_start|>assistant\n<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>',
 '<|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|im_start|>user\nWhat effect has Colonel Fitts had on his immediate family members?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nColonel Fitts has had a negative effect on his immediate family, particularly his wife, who is described as being "broken" by his actions.<|im_end|>',
 "<|im_start|>user\nWhat is the relationship between ti

In [44]:
tokenizer.decode(outputs[:, inputs['input_ids'].shape[1]:], skip_special_tokens=False)

['<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>',
 '<think>\n\n</think>\n\nColonel Fitts has had a negative effect on his immediate family, particularly his wife, who is described as being "broken" by his actions.<|im_end|>',
 "<think>\n\n</think>\n\nIn Bade's future civilization, time travel and space travel are considered separate and distinct technologies.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>",
 '<think>\n\n</think>\n\nThe personal conflict between **Mr. Wilson** and **Mr. Hargrove** drives the narrative.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|

In [33]:
for example in batch:
    print(generate_response([example['messages'][:-1]], model, tokenizer))

['']
['<think>\n\n</think>\n\nColonel Fitts has driven his family members to the brink of insanity.']
["<think>\n\n</think>\n\nIn Bade's future civilization, time travel and space travel are inextricably linked and considered the same phenomenon."]
['']


In [46]:
model_responses

['',
 "<think>\n\n</think>\n\nThe text provides several examples of behaviors that were considered bad sportsmanship in early UFC events, including:\n- Punching the referee\n- Punching the opponent's manager\n- Punching the cornerman\n- Punching the ring announcer\n- Punching the lights\n- Punching the air",
 '<think>\n\n</think>\n\nBroderick uses a strategy of offering them a chance to buy out the company, which is a financially attractive proposition.',
 "<think>\n\n</think>\n\nThe trouble at Mr. Ross's workplace serves as the central mystery, driving the investigation and forcing the family to travel to the town of Maitland.",
 '<think>\n\n</think>\n\nMrs. Deshazaway offers Fownes a deal: if he successfully gets them outside the house, she will let him have her as a "wife."',
 '<think>\n\n</think>\n\nThe development of human races was characterized by a "catastrophic acceleration" in the emergence of races, which occurred in a matter of months rather than the usual tens of thousands

In [23]:
# evaluation
import asyncio

semaphore = asyncio.Semaphore(64)

async def limited_judge_response(question, golden_answer, response, pbar):
    async with semaphore:
        judge_result = await judge_response(question, golden_answer, response)
        pbar.update(1)
        return judge_result



pbar = tqdm(total=len(test_data), ncols=80)
tasks = []
for item, response in zip(test_data, model_responses):
    q = item['messages'][0]['content']
    a = item['messages'][1]['content']
    tasks.append(limited_judge_response(q, a, response, pbar))

results = await asyncio.gather(*tasks, return_exceptions=True)

In [22]:
pbar.close()

 90%|█████████████████████████████████████▊    | 90/100 [01:21<00:09,  1.11it/s]


In [25]:
n_correct = 0
for x in results:
    if isinstance(x, bool) and x:
        n_correct += 1
n_correct / len(results)


0.04